# Marvin, o robô coletor de lixo

Trabalho Prático III de Fundamentos de Inteligência Artificial (2026/2), UFMG.

Este notebook treina Marvin com Q-Learning tabular e política ε-greedy, sem que o agente conheça o modelo do ambiente. A lógica fica nos módulos de `src/`, e aqui são executados os experimentos das Tarefas 2 a 5, com gráficos e tabelas.

Como executar: abra o notebook a partir da pasta `notebooks/` com o ambiente virtual ativo e rode as células em ordem. Os resultados de cada configuração são guardados em `resultados/dados/`, então a primeira execução completa leva cerca de 10 minutos e as seguintes são quase imediatas.

Roteiro:
1. Ambiente (Tarefa 2a)
2. Agente e agendas de ε (Tarefa 2b)
3. Teste de sensibilidade de α e γ
4. Treinamento e curvas de aprendizado (Tarefa 3)
5. Varredura de ε fixo (Tarefa 4a)
6. Agendas de decaimento (Tarefa 4b)
7. Política aprendida (Tarefa 5)

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# O notebook fica em notebooks/, então a raiz do projeto é a pasta acima
RAIZ_DO_PROJETO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ_DO_PROJETO))

from src.agendas_epsilon import criar_agenda_constante, criar_agenda_exponencial, criar_agenda_harmonica
from src.ambiente import Ambiente, decodificar_percepcao
from src.configuracao import Acao, ConfiguracaoTreinamento, numero_de_acoes, numero_de_estados
from src.experimentos import (
    analisar_sensibilidade,
    comparar_agendas,
    executar_configuracao,
    montar_tabela_de_sensibilidade,
    montar_tabela_resumo,
    tabela_em_latex,
    varredura_de_epsilon,
)
from src.graficos import (
    plotar_comparacao_de_configuracoes,
    plotar_curva_de_aprendizado,
    plotar_mapa_de_sensibilidade,
    plotar_metricas_de_diagnostico,
)
from src.politica import (
    montar_tabela_da_politica,
    resumir_politica,
    selecionar_estados_cercados,
    selecionar_estados_com_lixo,
    tabela_da_politica_em_latex,
)

In [ ]:
# Número de execuções com sementes diferentes, agregadas em cada gráfico e tabela
NUMERO_DE_EXECUCOES = 10
NUMERO_DE_EXECUCOES_SENSIBILIDADE = 5

# None usa todos os núcleos do computador, e 1 desativa o paralelismo
NUMERO_DE_PROCESSOS = None

PASTA_DE_DADOS = RAIZ_DO_PROJETO / "resultados" / "dados"
PASTA_DE_FIGURAS = RAIZ_DO_PROJETO / "resultados" / "figuras"
PASTA_DE_TABELAS = RAIZ_DO_PROJETO / "resultados" / "tabelas"
PASTA_DE_TABELAS.mkdir(parents=True, exist_ok=True)

## 1. Ambiente (Tarefa 2a)

A grade tem 50 × 50 células, com 15% de paredes, fixas, e 20% de lixo nas células livres, sorteado de novo a cada episódio. Marvin começa em (0, 0) e só percebe a célula atual e as quatro vizinhas. A figura mostra o canto superior esquerdo, como na Figura 2 do enunciado.

In [ ]:
def desenhar_grade(ambiente, tamanho_do_recorte=12):
    """Desenha o canto superior esquerdo da grade com a posição de Marvin.

    Args:
        ambiente: Ambiente a desenhar.
        tamanho_do_recorte: Quantidade de linhas e colunas mostradas.
    """
    recorte = ambiente.grade[:tamanho_do_recorte, :tamanho_do_recorte]
    mapa_de_cores = ListedColormap(["#444444", "white", "#008a17"])  # Parede, vazio e lixo

    figura, eixo = plt.subplots(figsize=(4.5, 4.5))
    eixo.imshow(recorte, cmap=mapa_de_cores, vmin=-1, vmax=1)
    coluna, linha = ambiente.posicao
    if coluna < tamanho_do_recorte and linha < tamanho_do_recorte:
        eixo.scatter([coluna], [linha], s=160, color="#257de0", zorder=3)

    eixo.set_xticks(range(tamanho_do_recorte))
    eixo.set_yticks(range(tamanho_do_recorte))
    eixo.set_xlabel("x (coluna)")
    eixo.set_ylabel("y (linha)")
    eixo.set_xticks(np.arange(-0.5, tamanho_do_recorte), minor=True)
    eixo.set_yticks(np.arange(-0.5, tamanho_do_recorte), minor=True)
    eixo.grid(which="minor", color="#cccccc", linewidth=0.8)
    eixo.tick_params(which="minor", length=0)
    legenda = [
        Patch(color="#257de0", label="Marvin"),
        Patch(color="#008a17", label="Lixo"),
        Patch(color="#444444", label="Parede"),
        Patch(facecolor="white", edgecolor="#cccccc", label="Vazio"),
    ]
    eixo.legend(handles=legenda, loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)
    plt.show()


ambiente = Ambiente(semente=42)
estado = ambiente.reset()
desenhar_grade(ambiente)
print("Percepção inicial (célula atual, norte, sul, leste, oeste):")
print([celula.name for celula in decodificar_percepcao(estado)])

In [ ]:
# Um passo de exemplo: tentar andar para o norte a partir de (0, 0) bate na borda
novo_estado, recompensa, terminado, resultado = ambiente.passo(Acao.norte)
print(f"Recompensa: {recompensa}, resultado: {resultado.name}, posição: {ambiente.posicao}")

## 2. Agente e agendas de ε (Tarefa 2b)

O agente guarda a Q-table (162 estados por 7 ações), escolhe ações pela política ε-greedy e atualiza as estimativas pela Eq. (2) do enunciado. A figura mostra as três agendas de decaimento comparadas na Tarefa 4(b), com ε₀ = 1,0 e ε mínimo de 0,01.

In [ ]:
print(f"Q-table: {numero_de_estados} estados x {numero_de_acoes} ações = {numero_de_estados * numero_de_acoes} entradas")

episodios = np.arange(1, 501)
agendas_de_epsilon = {
    "Constante (0,1)": criar_agenda_constante(0.1),
    "Exponencial (d = 0,99)": criar_agenda_exponencial(),
    "Harmônica (1/k)": criar_agenda_harmonica(),
}

figura, eixo = plt.subplots(figsize=(7, 3.4))
for nome, agenda in agendas_de_epsilon.items():
    eixo.plot(episodios, [agenda(episodio) for episodio in episodios], label=nome)
eixo.set_xlabel("Episódio k")
eixo.set_ylabel("ε")
eixo.set_title("Agendas de decaimento de ε", loc="left")
eixo.legend(frameon=False)
plt.show()

## 3. Teste de sensibilidade de α e γ

Antes de comparar estratégias de exploração, α, γ, K e H precisam ficar fixos. Para escolher α e γ, treina-se com a agenda exponencial para cada par da grade abaixo e compara-se a recompensa média dos últimos 50 episódios.

In [ ]:
sensibilidade = analisar_sensibilidade(
    numero_de_execucoes=NUMERO_DE_EXECUCOES_SENSIBILIDADE,
    numero_de_processos=NUMERO_DE_PROCESSOS,
    pasta_de_dados=PASTA_DE_DADOS,
)
tabela_de_sensibilidade = montar_tabela_de_sensibilidade(sensibilidade)
display(tabela_de_sensibilidade.round(1))

figura = plotar_mapa_de_sensibilidade(tabela_de_sensibilidade, PASTA_DE_FIGURAS / "sensibilidade.png")
plt.show()

**Escolha dos hiperparâmetros:** (preencher com a justificativa a partir da tabela e do mapa acima)

In [ ]:
# Hiperparâmetros congelados para todos os experimentos seguintes
# Ajuste os valores conforme o teste de sensibilidade
CONFIGURACAO_FINAL = replace(
    ConfiguracaoTreinamento(),
    taxa_de_aprendizado=0.1,
    fator_de_desconto=0.9,
)

ARGUMENTOS_DOS_EXPERIMENTOS = {
    "numero_de_execucoes": NUMERO_DE_EXECUCOES,
    "configuracao_treinamento": CONFIGURACAO_FINAL,
    "numero_de_processos": NUMERO_DE_PROCESSOS,
    "pasta_de_dados": PASTA_DE_DADOS,
}
print(CONFIGURACAO_FINAL)

## 4. Treinamento e curvas de aprendizado (Tarefa 3)

Marvin é treinado por K = 500 episódios de H = 1000 passos com a agenda exponencial, repetindo com sementes diferentes. A recompensa por episódio é ruidosa, então sobrepõe-se uma média móvel de 20 episódios. Os gráficos de diagnóstico mostram por que a recompensa subiu: mais lixos coletados e menos coletas malsucedidas e movimentos inválidos.

In [ ]:
treinamento_final = executar_configuracao(
    "Exponencial (d = 0,99)",
    "treinamento_final",
    "exponencial",
    {"epsilon_inicial": 1.0, "decaimento": 0.99, "epsilon_minimo": 0.01},
    **ARGUMENTOS_DOS_EXPERIMENTOS,
)

plotar_curva_de_aprendizado(
    treinamento_final.recompensa_por_episodio,
    "Curva de aprendizado, agenda exponencial",
    caminho_de_saida=PASTA_DE_FIGURAS / "curva_de_aprendizado.png",
)
plt.show()

plotar_metricas_de_diagnostico(
    treinamento_final, caminho_de_saida=PASTA_DE_FIGURAS / "metricas_de_diagnostico.png"
)
plt.show()

In [ ]:
# Tarefa 3c: fração de pares (s, a) visitados ao menos uma vez ao final do treinamento
fracoes_visitadas = treinamento_final.fracao_visitada_final_por_execucao
print(
    f"Pares (s, a) visitados: {100 * fracoes_visitadas.mean():.1f}% "
    f"(desvio padrão {100 * fracoes_visitadas.std(ddof=1):.1f} p.p., n = {len(fracoes_visitadas)} execuções)"
)

**Análise da curva de aprendizado:** (preencher)

## 5. Varredura de ε fixo (Tarefa 4a)

Com α, γ, K e H fixos, treina-se com ε ∈ {0; 0,01; 0,05; 0,1; 0,3; 1,0}. A tabela traz a recompensa média dos últimos 50 episódios, o melhor episódio e a porcentagem da Q-table visitada, em média e desvio padrão entre as execuções.

In [ ]:
varredura = varredura_de_epsilon(**ARGUMENTOS_DOS_EXPERIMENTOS)
tabela_da_varredura = montar_tabela_resumo(varredura)
display(tabela_da_varredura.round(1))

(PASTA_DE_TABELAS / "varredura_de_epsilon.tex").write_text(
    tabela_em_latex(tabela_da_varredura, "Configuração"), encoding="utf-8"
)
plotar_comparacao_de_configuracoes(
    varredura, "Varredura de ε fixo", caminho_de_saida=PASTA_DE_FIGURAS / "varredura_de_epsilon.png"
)
plt.show()

**Análise dos extremos:** (explicar, com os números da tabela, por que ε = 0 e ε = 1 falham por motivos opostos)

## 6. Agendas de decaimento (Tarefa 4b)

Três agendas com ε mínimo de 0,01: constante (0,1), exponencial (ε₀ = 1,0 e d = 0,99) e harmônica (1/k). As curvas de aprendizado ficam no mesmo gráfico.

In [ ]:
agendas_comparadas = comparar_agendas(**ARGUMENTOS_DOS_EXPERIMENTOS)
tabela_das_agendas = montar_tabela_resumo(agendas_comparadas)
display(tabela_das_agendas.round(1))

(PASTA_DE_TABELAS / "comparacao_de_agendas.tex").write_text(
    tabela_em_latex(tabela_das_agendas, "Agenda"), encoding="utf-8"
)
plotar_comparacao_de_configuracoes(
    agendas_comparadas,
    "Agendas de decaimento de ε",
    caminho_de_saida=PASTA_DE_FIGURAS / "comparacao_de_agendas.png",
)
plt.show()

**Qual agenda é melhor e em que fase do treinamento:** (preencher)

## 7. Política aprendida (Tarefa 5)

Cada execução gera uma Q-table própria, então a política é obtida por voto majoritário entre as execuções, considerando só os estados visitados. Estados nunca visitados não têm ação, pois o desempate do argmax não carrega informação. A configuração usada é a escolhida como melhor.

In [ ]:
# Troque pela configuração que tiver o melhor desempenho nas Tarefas 4a e 4b
configuracao_da_politica = agendas_comparadas["Exponencial (d = 0,99)"]

tabela_da_politica = montar_tabela_da_politica(
    configuracao_da_politica.q_tables, configuracao_da_politica.contagem_de_visitas
)
colunas = ["celula_atual", "norte", "sul", "leste", "oeste", "acao", "execucoes_que_visitaram", "concordancia"]
print(f"Estados visitados: {(tabela_da_politica['acao'] != 'não visitado').sum()} de {numero_de_estados}")

print("\nEstados com lixo na célula atual:")
estados_com_lixo = selecionar_estados_com_lixo(tabela_da_politica)
display(estados_com_lixo[colunas].round(2))

In [ ]:
# Estados com pelo menos 3 direções bloqueadas por parede (o enunciado pede uma ou mais)
print("Estados cercados por paredes em 3 ou mais direções:")
estados_cercados = selecionar_estados_cercados(tabela_da_politica, minimo_de_paredes=3)
display(estados_cercados[colunas].round(2))

In [ ]:
# Resumo do comportamento da política, medido só nos estados visitados
descricoes = {
    "coleta_quando_ha_lixo_na_celula": "Coleta quando há lixo na célula atual",
    "anda_para_lixo_vizinho": "Anda em direção a um lixo vizinho",
    "evita_direcao_com_parede": "Não escolhe direção com parede",
}
for chave, (fracao, quantidade) in resumir_politica(tabela_da_politica).items():
    print(f"{descricoes[chave]}: {100 * fracao:.1f}% ({quantidade} estados)")

(PASTA_DE_TABELAS / "politica_estados_cercados.tex").write_text(
    tabela_da_politica_em_latex(estados_cercados), encoding="utf-8"
)

**Análise da política aprendida:** (preencher com o comportamento observado e as limitações, como a ambiguidade perceptual e a ausência de memória)